# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sidd13789/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [26]:
%pip -q install duckdb huggingface_hub

In [27]:
import os
import getpass
import duckdb

HF_TOKEN = os.environ.get("HF_TOKEN") or getpass.getpass("Paste your Hugging Face READ token: ")

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "dim_clients": f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content": f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily": f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_daily_sample": f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    "fact_query_90d": f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

print("✅ Connected Successfully")

Paste your Hugging Face READ token: ··········
✅ Connected Successfully


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [28]:
print("Unit of Analysis")
print("One row = One content item (content_hash_id) for one report_date.")
print("Time window = March 2026 (month = '2026-03').")

Unit of Analysis
One row = One content item (content_hash_id) for one report_date.
Time window = March 2026 (month = '2026-03').


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [29]:
contract = {
    "Feature": [
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position"
    ],
    "Label": [
        "is_declining"
    ],
    "Context": [
        "client_hash_id",
        "content_hash_id",
        "report_date"
    ],
    "Excluded": {
        "Future information": "Excluded because it would cause data leakage."
    }
}

contract

{'Feature': ['gsc_impressions', 'gsc_clicks', 'gsc_avg_position'],
 'Label': ['is_declining'],
 'Context': ['client_hash_id', 'content_hash_id', 'report_date'],
 'Excluded': {'Future information': 'Excluded because it would cause data leakage.'}}

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [30]:
con.sql(f"""
SELECT COUNT(*)
FROM {TABLES['dim_clients']}
""").df()

,count_star()
0,104


In [31]:
con.sql(f"""
DESCRIBE
SELECT *
FROM {TABLES['fact_daily']}
LIMIT 5
""").df()

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [32]:
con.sql(f"""
SELECT *
FROM {TABLES['fact_daily']}
WHERE content_hash_id = 'content_0a21add649629840'
  AND report_date = DATE '2026-06-20'
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-06-20,client_1a730cb2640a1abf,content_0a21add649629840,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
1,2026-06-20,client_1a730cb2640a1abf,content_0a21add649629840,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06


In [33]:
con.sql(f"""
SELECT
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date,
    COUNT(*) AS total_rows
FROM {TABLES['fact_daily']}
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,start_date,end_date,total_rows
0,2025-01-27,2026-06-30,78835655


### Observations

- The dataset was verified using SQL queries.
- The selected month for analysis is **2026-03**.
- The data contract was validated by checking:
  - grain,
  - row count and date range,
  - data availability (`IS TRUE`).

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This dataset has the following limitations:

- The history is unbalanced because different clients started collecting data at different times.
- Early dates may contain only Google Search Console (GSC) data, while GA4 data may not yet be available.
- Time windows can overlap, so care must be taken when creating training labels and features to avoid data leakage.
- Some records have missing values or unavailable GSC/GA4 data, reducing the number of usable observations.
- The dataset is anonymized, so it cannot identify real clients, URLs, or search queries.
- This dataset supports decision-making but cannot prove cause-and-effect relationships.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.